In [3]:
# !pip install selenium webdriver-manager beautifulsoup4 pandas lxml

import re
import json
import time
from pathlib import Path

import pandas as pd
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager


# ----------------------------
# CONFIG
# ----------------------------
TD_CARDS = [
    {
        "card_id": "td_cash_back_visa_infinite_ca",
        "url": "https://www.td.com/ca/en/personal-banking/products/credit-cards/cash-back/cash-back-visa-infinite-card"
    },
    {
        "card_id": "td_cash_back_visa_ca",
        "url": "https://www.td.com/ca/en/personal-banking/products/credit-cards/cash-back/cash-back-visa-card"
    },
    {
        "card_id": "td_aeroplan_visa_platinum_card_ca",
        "url": "https://www.td.com/ca/en/personal-banking/products/credit-cards/aeroplan/aeroplan-visa-platinum-card"
    },
    {
        "card_id": "td_aeroplan_visa_infinite_privilege_card_ca",
        "url": "hhttps://www.td.com/ca/en/personal-banking/products/credit-cards/aeroplan/aeroplan-visa-infinite-privilege-card"
    },
    {
        "card_id": "td_first_class_travel_visa_infinite_card_ca",
        "url": "https://www.td.com/ca/en/personal-banking/products/credit-cards/travel-rewards/first-class-travel-visa-infinite-card"
    },
    {
        "card_id": "td_platinum_travel_visa_card_ca",
        "url": "https://www.td.com/ca/en/personal-banking/products/credit-cards/travel-rewards/platinum-travel-visa-card"
    },
    {
        "card_id": "td_rewards_visa_card_ca",
        "url": "https://www.td.com/ca/en/personal-banking/products/credit-cards/travel-rewards/rewards-visa-card"
    },
    {
        "card_id": "td_low_rate_visa_card_ca",
        "url": "https://www.td.com/ca/en/personal-banking/products/credit-cards/low-rate/low-rate-visa-card"
    },
    {
        "card_id": "td_us_dollar_visa_card_ca",
        "url": "https://www.td.com/ca/en/personal-banking/products/credit-cards/us-dollar/us-dollar-visa-card"
    },
    {
        "card_id": "td_business_travel_visa_card_ca",
        "url": "https://www.td.com/ca/en/business-banking/small-business/credit-cards/business-travel-visa-card"
    },
    {
        "card_id": "td_aeroplan_visa_business_card_ca",
        "url": "https://www.td.com/ca/en/business-banking/small-business/credit-cards/aeroplan-visa-business-card"
    },
    {
        "card_id": "td_business_cash_back_visa_card_ca",
        "url": "https://www.td.com/ca/en/business-banking/small-business/credit-cards/business-cash-back-visa-card"
    },
    {
        "card_id": "td_business_select_rate_visa_card_ca",
        "url": "https://www.td.com/ca/en/business-banking/small-business/credit-cards/business-select-rate-visa-card"
    }
]

DATA_DIR = Path(r"C:\Users\91951\OneDrive\Desktop\pythonProject\leetcode\Ai-ML-Projects\credit-card-rag\data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

CSV_PATH = DATA_DIR / "cards_min_td.csv"
SNIPPETS_PATH = DATA_DIR / "card_snippets_td.jsonl"


# ----------------------------
# BROWSER
# ----------------------------
def get_driver(headless: bool = True):
    chrome_options = Options()
    if headless:
        chrome_options.add_argument("--headless=new")
    chrome_options.add_argument("--window-size=1700,3000")
    chrome_options.add_argument("--disable-blink-features=AutomationControlled")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")

    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=chrome_options
    )
    return driver


# ----------------------------
# HELPERS
# ----------------------------
def clean_text(text: str) -> str:
    if not text:
        return ""
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def search_first(patterns, text, flags=re.IGNORECASE):
    for pattern in patterns:
        m = re.search(pattern, text, flags)
        if m:
            return clean_text(m.group(1) if m.groups() else m.group(0))
    return ""


def normalize_money(value: str) -> str:
    if not value:
        return ""
    return value.replace("$", "").replace(",", "").strip()


def extract_between(text: str, start_markers, end_markers=None, max_chars=2500):
    lower = text.lower()
    start_idx = -1

    for marker in start_markers:
        idx = lower.find(marker.lower())
        if idx != -1:
            start_idx = idx
            break

    if start_idx == -1:
        return ""

    end_idx = len(text)
    if end_markers:
        for marker in end_markers:
            idx = lower.find(marker.lower(), start_idx + 1)
            if idx != -1:
                end_idx = min(end_idx, idx)

    return clean_text(text[start_idx:end_idx][:max_chars])


def infer_best_for(page_text: str):
    t = page_text.lower()
    tags = []

    if "cash back" in t:
        tags.append("cashback")

    if any(x in t for x in [
        "grocery", "gas", "electric vehicle charging", "public transit",
        "recurring bill payments", "streaming", "digital gaming"
    ]):
        tags.append("daily_spend")

    if "roadside assistance" in t or "auto club" in t:
        tags.append("automotive_perks")

    if "visa infinite" in t:
        tags.append("premium_perks")

    if "travel benefits" in t:
        tags.append("travel")

    seen = set()
    result = []
    for x in tags:
        if x not in seen:
            seen.add(x)
            result.append(x)
    return ",".join(result)


def infer_one_liner(page_text: str):
    return "Canadian cash back credit card for groceries, gas, transit, recurring bills, streaming, and premium Visa Infinite benefits."


def fetch_page_text(driver, url: str, wait_sec: int = 8):
    driver.get(url)
    time.sleep(wait_sec)

    html = driver.page_source
    soup = BeautifulSoup(html, "lxml")
    page_text = clean_text(soup.get_text(" ", strip=True))
    return html, soup, page_text


# ----------------------------
# TD EXTRACTION
# ----------------------------
def extract_td_card_name(page_text: str):
    return (
        search_first(
            [
                r"(TD Cash Back Visa Infinite\*?\s*Card)",
                r"(TD Cash Back Visa Infinite)"
            ],
            page_text
        )
        .replace("*", "")
        .strip()
        or "TD Cash Back Visa Infinite Card"
    )


def extract_td_annual_fee(page_text: str):
    fee = search_first(
        [
            r"Annual fee\s*\$([0-9]+(?:\.[0-9]{1,2})?)",
            r"Annual Fee[:\s]*\$([0-9]+(?:\.[0-9]{1,2})?)"
        ],
        page_text
    )
    return normalize_money(fee)


def extract_td_purchase_rate(page_text: str):
    return search_first(
        [
            r"Interest:\s*Purchases\s*([0-9]+(?:\.[0-9]{1,2})?%)",
            r"Purchase Rate[:\s]*([0-9]+(?:\.[0-9]{1,2})?%)"
        ],
        page_text
    )


def extract_td_cash_advance_rate(page_text: str):
    return search_first(
        [
            r"Interest:\s*Cash Advances\s*([0-9]+(?:\.[0-9]{1,2})?%)",
            r"Cash Advance Rate[:\s]*([0-9]+(?:\.[0-9]{1,2})?%)"
        ],
        page_text
    )


def extract_td_additional_card_fee(page_text: str):
    fee = search_first(
        [
            r"Additional Cardholder\s*[0-9]*\s*\$([0-9]+(?:\.[0-9]{1,2})?)",
            r"Additional Cardholder[:\s]*\$([0-9]+(?:\.[0-9]{1,2})?)"
        ],
        page_text
    )
    return normalize_money(fee)


def extract_td_additional_card_fee_detail(page_text: str):
    return search_first(
        [
            r"(\$[0-9]+\s*for first Additional Cardholder;\s*\$[0-9]+\s*for subsequent Additional Cardholders\.)"
        ],
        page_text
    )


def extract_td_welcome_bonus(page_text: str):
    return search_first(
        [
            r"(Earn up to \$[0-9,]+ in total value.*?Conditions apply\.)",
            r"(Earn up to \$[0-9,]+ in total value.*?Annual Fee rebate for the first year.*?\.)"
        ],
        page_text
    )


def extract_td_eligibility(page_text: str):
    income_line = search_first(
        [
            r"(\$[0-9,]+\s*personal or \$[0-9,]+\s*household annual income\.)"
        ],
        page_text
    )

    resident_line = search_first(
        [
            r"(You are a Canadian resident and are of the age of majority in your province/territory of residence\.)"
        ],
        page_text
    )

    parts = [x for x in [income_line, resident_line] if x]
    return clean_text(" | ".join(parts))


def extract_td_rewards_summary(page_text: str):
    three_pct_1 = search_first(
        [
            r"(Earn 3% in Cash Back Dollars on Grocery, Gas & Electric Vehicle Charging, and Public Transit Purchases[^\.]*\.)"
        ],
        page_text
    )

    three_pct_2 = search_first(
        [
            r"(Earn 3% in Cash Back Dollars on Recurring Bill Payments, and Streaming, Digital Gaming & Media Purchases[^\.]*\.)"
        ],
        page_text
    )

    one_pct = search_first(
        [
            r"(Earn 1% in Cash Back Dollars on all other Purchases[^\.]*\.)"
        ],
        page_text
    )

    parts = [x for x in [three_pct_1, three_pct_2, one_pct] if x]
    return clean_text(" | ".join(parts))


def extract_td_redemption_summary(page_text: str):
    return search_first(
        [
            r"(Redeem your Cash Back Dollars.*?the choice is yours[^\.!]*[.!])",
            r"(Redeeming Cash Back dollars is simple and easy\..*?good standing with TD[^\.!]*[.!])"
        ],
        page_text
    )


def scrape_td_card(driver, card_cfg):
    url = card_cfg["url"]
    card_id = card_cfg["card_id"]

    html, soup, page_text = fetch_page_text(driver, url)

    card_name = extract_td_card_name(page_text)
    issuer = "TD"
    country = "Canada"
    network = "Visa"
    card_type = "Credit Card"

    annual_fee = extract_td_annual_fee(page_text)
    purchase_rate = extract_td_purchase_rate(page_text)
    cash_advance_rate = extract_td_cash_advance_rate(page_text)
    additional_card_fee = extract_td_additional_card_fee(page_text)
    additional_card_fee_detail = extract_td_additional_card_fee_detail(page_text)

    welcome_bonus_summary = extract_td_welcome_bonus(page_text)
    eligibility_summary = extract_td_eligibility(page_text)
    rewards_summary = extract_td_rewards_summary(page_text)
    redemption_summary = extract_td_redemption_summary(page_text)

    best_for = infer_best_for(page_text)
    one_liner = infer_one_liner(page_text)

    # Main chunks
    overview_text = extract_between(
        page_text,
        start_markers=["TD Cash Back Visa Infinite Card", "TD Cash Back Visa Infinite"],
        end_markers=["Eligibility Requirements:", "At a glance"],
        max_chars=1600
    )

    eligibility_text = extract_between(
        page_text,
        start_markers=["Eligibility Requirements:"],
        end_markers=["Special Offer", "At a glance"],
        max_chars=1000
    )

    at_a_glance_text = extract_between(
        page_text,
        start_markers=["At a glance"],
        end_markers=["It's easy to earn and redeem Cash Back Dollars", "Perks and offers"],
        max_chars=1200
    )

    rewards_text = extract_between(
        page_text,
        start_markers=["It's easy to earn and redeem Cash Back Dollars"],
        end_markers=["Perks and offers", "More Card Benefits and Features"],
        max_chars=1800
    )

    perks_text = extract_between(
        page_text,
        start_markers=["Perks and offers"],
        end_markers=["More Card Benefits and Features", "Frequently asked questions"],
        max_chars=1200
    )

    # Very important for vector DB
    visa_infinite_text = extract_between(
        page_text,
        start_markers=["Visa Infinite Benefits"],
        end_markers=["Automotive Benefits"],
        max_chars=1500
    )

    automotive_text = extract_between(
        page_text,
        start_markers=["Automotive Benefits"],
        end_markers=["Travel Benefits"],
        max_chars=1500
    )

    travel_text = extract_between(
        page_text,
        start_markers=["Travel Benefits"],
        end_markers=["Additional Benefits & Features"],
        max_chars=1500
    )

    additional_features_text = extract_between(
        page_text,
        start_markers=["Additional Benefits & Features"],
        end_markers=["Security"],
        max_chars=1500
    )

    security_text = extract_between(
        page_text,
        start_markers=["Security"],
        end_markers=["Optional benefits"],
        max_chars=1500
    )

    optional_benefits_text = extract_between(
        page_text,
        start_markers=["Optional benefits"],
        end_markers=["Frequently asked questions", "Why should you get the TD Cash Back Visa Infinite Credit Card?"],
        max_chars=1500
    )

    faq_why_text = extract_between(
        page_text,
        start_markers=["Why should you get the TD Cash Back Visa Infinite Credit Card?"],
        end_markers=["How do you apply for this Cash Back credit card?"],
        max_chars=1200
    )

    faq_apply_text = extract_between(
        page_text,
        start_markers=["How do you apply for this Cash Back credit card?"],
        end_markers=["How do you earn Cash Back on this TD Cash Back Credit Card?"],
        max_chars=1000
    )

    faq_earn_text = extract_between(
        page_text,
        start_markers=["How do you earn Cash Back on this TD Cash Back Credit Card?"],
        end_markers=["How do you redeem Cash Back dollars?"],
        max_chars=1000
    )

    faq_redeem_text = extract_between(
        page_text,
        start_markers=["How do you redeem Cash Back dollars?"],
        end_markers=["Find out more about our credit cards", "Legal"],
        max_chars=1000
    )

    fees_text = clean_text(
        f"Annual fee: ${annual_fee}; "
        f"Purchase interest: {purchase_rate}; "
        f"Cash advance interest: {cash_advance_rate}; "
        f"Additional cardholder fee: ${additional_card_fee}. "
        f"{additional_card_fee_detail}"
    )

    card_record = {
        "card_id": card_id,
        "card_name": card_name,
        "issuer": issuer,
        "country": country,
        "network": network,
        "card_type": card_type,
        "link": url,
        "monthly_fee": "",
        "annual_fee": annual_fee,
        "welcome_bonus_summary": welcome_bonus_summary,
        "best_for": best_for,
        "one_liner": one_liner,
        "eligibility_summary": eligibility_summary,

        # useful extras
        "purchase_rate": purchase_rate,
        "cash_advance_rate": cash_advance_rate,
        "additional_card_fee": additional_card_fee,
        "additional_card_fee_detail": additional_card_fee_detail,
        "rewards_summary": rewards_summary,
        "redemption_summary": redemption_summary
    }

    snippets = [
        {"chunk_id": f"{card_id}_overview", "card_id": card_id, "section": "overview", "text": overview_text},
        {"chunk_id": f"{card_id}_eligibility", "card_id": card_id, "section": "eligibility", "text": eligibility_text},
        {"chunk_id": f"{card_id}_at_a_glance", "card_id": card_id, "section": "at_a_glance", "text": at_a_glance_text},
        {"chunk_id": f"{card_id}_welcome_bonus", "card_id": card_id, "section": "welcome_bonus", "text": welcome_bonus_summary},
        {"chunk_id": f"{card_id}_rewards", "card_id": card_id, "section": "rewards", "text": rewards_text},
        {"chunk_id": f"{card_id}_perks", "card_id": card_id, "section": "perks_and_offers", "text": perks_text},
        {"chunk_id": f"{card_id}_visa_infinite", "card_id": card_id, "section": "visa_infinite_benefits", "text": visa_infinite_text},
        {"chunk_id": f"{card_id}_automotive", "card_id": card_id, "section": "automotive_benefits", "text": automotive_text},
        {"chunk_id": f"{card_id}_travel", "card_id": card_id, "section": "travel_benefits", "text": travel_text},
        {"chunk_id": f"{card_id}_additional_features", "card_id": card_id, "section": "additional_features", "text": additional_features_text},
        {"chunk_id": f"{card_id}_security", "card_id": card_id, "section": "security", "text": security_text},
        {"chunk_id": f"{card_id}_optional", "card_id": card_id, "section": "optional_benefits", "text": optional_benefits_text},
        {"chunk_id": f"{card_id}_faq_why", "card_id": card_id, "section": "faq_why_get_this_card", "text": faq_why_text},
        {"chunk_id": f"{card_id}_faq_apply", "card_id": card_id, "section": "faq_apply", "text": faq_apply_text},
        {"chunk_id": f"{card_id}_faq_earn", "card_id": card_id, "section": "faq_earn", "text": faq_earn_text},
        {"chunk_id": f"{card_id}_faq_redeem", "card_id": card_id, "section": "faq_redeem", "text": faq_redeem_text},
        {"chunk_id": f"{card_id}_fees", "card_id": card_id, "section": "fees", "text": fees_text},
    ]

    # remove empty snippets
    snippets = [s for s in snippets if s["text"]]

    return card_record, snippets


# ----------------------------
# RUN
# ----------------------------
driver = get_driver(headless=True)

all_cards = []
all_snippets = []

try:
    for card in TD_CARDS:
        print(f"Scraping: {card['card_id']}")
        card_record, snippets = scrape_td_card(driver, card)
        all_cards.append(card_record)
        all_snippets.extend(snippets)
        time.sleep(2)
finally:
    driver.quit()


# ----------------------------
# SAVE
# ----------------------------
cards_df = pd.DataFrame(all_cards)
cards_df.to_csv(CSV_PATH, index=False)

with open(SNIPPETS_PATH, "w", encoding="utf-8") as f:
    for item in all_snippets:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print("Saved CSV to:", CSV_PATH)
print("Saved snippets to:", SNIPPETS_PATH)
print(cards_df.T)

print("\nSnippet preview:")
for s in all_snippets:
    print(f"- {s['section']}: {s['text'][:220]}...")

Scraping: td_cash_back_visa_infinite_ca
Scraping: td_cash_back_visa_ca
Scraping: td_aeroplan_visa_platinum_card_ca
Scraping: td_aeroplan_visa_infinite_privilege_card_ca
Scraping: td_first_class_travel_visa_infinite_card_ca
Scraping: td_platinum_travel_visa_card_ca
Scraping: td_rewards_visa_card_ca
Scraping: td_low_rate_visa_card_ca
Scraping: td_us_dollar_visa_card_ca
Scraping: td_business_travel_visa_card_ca
Scraping: td_aeroplan_visa_business_card_ca
Scraping: td_business_cash_back_visa_card_ca
Scraping: td_business_select_rate_visa_card_ca
Saved CSV to: C:\Users\91951\OneDrive\Desktop\pythonProject\leetcode\Ai-ML-Projects\credit-card-rag\data\cards_min_td.csv
Saved snippets to: C:\Users\91951\OneDrive\Desktop\pythonProject\leetcode\Ai-ML-Projects\credit-card-rag\data\card_snippets_td.jsonl
                                                                           0   \
card_id                                         td_cash_back_visa_infinite_ca   
card_name                          

In [4]:
# !pip install selenium webdriver-manager beautifulsoup4 pandas lxml

import re
import json
import time
from pathlib import Path

import pandas as pd
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager


# ----------------------------
# CONFIG
# ----------------------------
TD_CARDS = [
    {
        "card_id": "td_cash_back_visa_infinite_ca",
        "url": "https://www.td.com/ca/en/personal-banking/products/credit-cards/cash-back/cash-back-visa-infinite-card"
    },
    {
        "card_id": "td_cash_back_visa_ca",
        "url": "https://www.td.com/ca/en/personal-banking/products/credit-cards/cash-back/cash-back-visa-card"
    },
    {
        "card_id": "td_aeroplan_visa_platinum_card_ca",
        "url": "https://www.td.com/ca/en/personal-banking/products/credit-cards/aeroplan/aeroplan-visa-platinum-card"
    },
    {
        "card_id": "td_aeroplan_visa_infinite_privilege_card_ca",
        "url": "hhttps://www.td.com/ca/en/personal-banking/products/credit-cards/aeroplan/aeroplan-visa-infinite-privilege-card"
    },
    {
        "card_id": "td_first_class_travel_visa_infinite_card_ca",
        "url": "https://www.td.com/ca/en/personal-banking/products/credit-cards/travel-rewards/first-class-travel-visa-infinite-card"
    },
    {
        "card_id": "td_platinum_travel_visa_card_ca",
        "url": "https://www.td.com/ca/en/personal-banking/products/credit-cards/travel-rewards/platinum-travel-visa-card"
    },
    {
        "card_id": "td_rewards_visa_card_ca",
        "url": "https://www.td.com/ca/en/personal-banking/products/credit-cards/travel-rewards/rewards-visa-card"
    },
    {
        "card_id": "td_low_rate_visa_card_ca",
        "url": "https://www.td.com/ca/en/personal-banking/products/credit-cards/low-rate/low-rate-visa-card"
    },
    {
        "card_id": "td_us_dollar_visa_card_ca",
        "url": "https://www.td.com/ca/en/personal-banking/products/credit-cards/us-dollar/us-dollar-visa-card"
    },
    {
        "card_id": "td_business_travel_visa_card_ca",
        "url": "https://www.td.com/ca/en/business-banking/small-business/credit-cards/business-travel-visa-card"
    },
    {
        "card_id": "td_aeroplan_visa_business_card_ca",
        "url": "https://www.td.com/ca/en/business-banking/small-business/credit-cards/aeroplan-visa-business-card"
    },
    {
        "card_id": "td_business_cash_back_visa_card_ca",
        "url": "https://www.td.com/ca/en/business-banking/small-business/credit-cards/business-cash-back-visa-card"
    },
    {
        "card_id": "td_business_select_rate_visa_card_ca",
        "url": "https://www.td.com/ca/en/business-banking/small-business/credit-cards/business-select-rate-visa-card"
    }
]

DATA_DIR = Path(r"C:\Users\91951\OneDrive\Desktop\pythonProject\leetcode\Ai-ML-Projects\credit-card-rag\data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

CSV_PATH = DATA_DIR / "cards_min_td.csv"
SNIPPETS_PATH = DATA_DIR / "card_snippets_td.jsonl"


# ----------------------------
# BROWSER
# ----------------------------
def get_driver(headless: bool = True):
    chrome_options = Options()
    if headless:
        chrome_options.add_argument("--headless=new")
    chrome_options.add_argument("--window-size=1800,3200")
    chrome_options.add_argument("--disable-blink-features=AutomationControlled")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")

    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=chrome_options
    )
    return driver


# ----------------------------
# HELPERS
# ----------------------------
def clean_text(text: str) -> str:
    if not text:
        return ""
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def search_first(patterns, text, flags=re.IGNORECASE):
    for pattern in patterns:
        m = re.search(pattern, text, flags)
        if m:
            return clean_text(m.group(1) if m.groups() else m.group(0))
    return ""


def normalize_money(value: str) -> str:
    if not value:
        return ""
    return value.replace("$", "").replace(",", "").strip()


def extract_between(text: str, start_markers, end_markers=None, max_chars=2500):
    lower = text.lower()
    start_idx = -1

    for marker in start_markers:
        idx = lower.find(marker.lower())
        if idx != -1:
            start_idx = idx
            break

    if start_idx == -1:
        return ""

    end_idx = len(text)
    if end_markers:
        for marker in end_markers:
            idx = lower.find(marker.lower(), start_idx + 1)
            if idx != -1:
                end_idx = min(end_idx, idx)

    return clean_text(text[start_idx:end_idx][:max_chars])


def infer_best_for(page_text: str):
    t = page_text.lower()
    tags = []

    if "cash back" in t:
        tags.append("cashback")
    if any(x in t for x in [
        "grocery", "gas", "electric vehicle charging", "public transit",
        "recurring bill payments", "streaming", "digital gaming", "media purchases"
    ]):
        tags.append("daily_spend")
    if "roadside assistance" in t or "auto club" in t:
        tags.append("automotive_perks")
    if "visa infinite" in t:
        tags.append("premium_perks")
    if "travel benefits" in t:
        tags.append("travel")

    seen = set()
    result = []
    for tag in tags:
        if tag not in seen:
            seen.add(tag)
            result.append(tag)
    return ",".join(result)


def infer_one_liner(page_text: str):
    return "Canadian cash back credit card for groceries, gas, transit, recurring bills, streaming, and Visa Infinite benefits."


def fetch_page_text(driver, url: str, wait_sec: int = 10):
    driver.get(url)
    time.sleep(wait_sec)

    html = driver.page_source
    soup = BeautifulSoup(html, "lxml")
    page_text = clean_text(soup.get_text(" ", strip=True))
    return html, soup, page_text


# ----------------------------
# CARD NAME FIX
# ----------------------------
def extract_generic_td_card_name(driver, soup, page_text: str):
    # 1. Title tag
    if soup.title and soup.title.text:
        title = clean_text(soup.title.text).replace("*", "")
        title = title.split("|")[0].strip()

        if "TD" in title and ("Card" in title or "Visa" in title or "Mastercard" in title):
            return title

    # 2. Visible heading candidates
    candidate_selectors = [
        "h1",
        "[data-td-cm='product-name']",
        ".cmp-product-details__title",
        ".product-name",
        ".cc-card-title",
        ".card-title",
    ]

    for selector in candidate_selectors:
        try:
            elems = driver.find_elements(By.CSS_SELECTOR, selector)
            for elem in elems:
                txt = clean_text(elem.text).replace("*", "")
                if txt and "TD" in txt and ("Card" in txt or "Visa" in txt or "Mastercard" in txt):
                    return txt
        except Exception:
            pass

    # 3. Text fallback
    patterns = [
        r"(TD [A-Za-z0-9\s\-\+&/]+ Visa Infinite\*?\s*Card)",
        r"(TD [A-Za-z0-9\s\-\+&/]+ Visa\*?\s*Card)",
        r"(TD [A-Za-z0-9\s\-\+&/]+ Mastercard\*?\s*Card)",
        r"(TD [A-Za-z0-9\s\-\+&/]+ Card)"
    ]
    for pattern in patterns:
        m = re.search(pattern, page_text, re.IGNORECASE)
        if m:
            return clean_text(m.group(1)).replace("*", "")

    return ""


# ----------------------------
# TD FIELD EXTRACTION
# ----------------------------
def extract_td_annual_fee(page_text: str):
    fee = search_first(
        [
            r"Annual fee\s*\$([0-9]+(?:\.[0-9]{1,2})?)",
            r"Annual Fee[:\s]*\$([0-9]+(?:\.[0-9]{1,2})?)"
        ],
        page_text
    )
    return normalize_money(fee)


def extract_td_purchase_rate(page_text: str):
    return search_first(
        [
            r"Interest:\s*Purchases\s*([0-9]+(?:\.[0-9]{1,2})?%)",
            r"Purchase Rate[:\s]*([0-9]+(?:\.[0-9]{1,2})?%)"
        ],
        page_text
    )


def extract_td_cash_advance_rate(page_text: str):
    return search_first(
        [
            r"Interest:\s*Cash Advances\s*([0-9]+(?:\.[0-9]{1,2})?%)",
            r"Cash Advance Rate[:\s]*([0-9]+(?:\.[0-9]{1,2})?%)"
        ],
        page_text
    )


def extract_td_additional_card_fee(page_text: str):
    fee = search_first(
        [
            r"Additional Cardholder\s*[0-9]*\s*\$([0-9]+(?:\.[0-9]{1,2})?)",
            r"Additional Cardholder[:\s]*\$([0-9]+(?:\.[0-9]{1,2})?)"
        ],
        page_text
    )
    return normalize_money(fee)


def extract_td_additional_card_fee_detail(page_text: str):
    return search_first(
        [
            r"(\$[0-9]+\s*for first Additional Cardholder;\s*\$[0-9]+\s*for subsequent Additional Cardholders\.)"
        ],
        page_text
    )


def extract_td_welcome_bonus(page_text: str):
    return search_first(
        [
            r"(Earn up to \$[0-9,]+ in total value.*?Conditions apply\.)",
            r"(Earn up to \$[0-9,]+ in total value.*?Annual Fee rebate for the first year.*?\.)"
        ],
        page_text
    )


def extract_td_eligibility(page_text: str):
    income_line = search_first(
        [
            r"(\$[0-9,]+\s*personal or \$[0-9,]+\s*household annual income\.)"
        ],
        page_text
    )

    resident_line = search_first(
        [
            r"(You are a Canadian resident and are of the age of majority in your province/territory of residence\.)"
        ],
        page_text
    )

    parts = [x for x in [income_line, resident_line] if x]
    return clean_text(" | ".join(parts))


def extract_td_rewards_summary(page_text: str):
    three_pct_1 = search_first(
        [
            r"(Earn 3% in Cash Back Dollars on Grocery, Gas & Electric Vehicle Charging, and Public Transit Purchases[^\.]*\.)"
        ],
        page_text
    )

    three_pct_2 = search_first(
        [
            r"(Earn 3% in Cash Back Dollars on Recurring Bill Payments, and Streaming, Digital Gaming & Media Purchases[^\.]*\.)"
        ],
        page_text
    )

    one_pct = search_first(
        [
            r"(Earn 1% in Cash Back Dollars on all other Purchases[^\.]*\.)"
        ],
        page_text
    )

    parts = [x for x in [three_pct_1, three_pct_2, one_pct] if x]
    return clean_text(" | ".join(parts))


def extract_td_redemption_summary(page_text: str):
    return search_first(
        [
            r"(Redeem your Cash Back Dollars.*?the choice is yours[^\.!]*[.!])",
            r"(Redeem any time.*?choice is yours[^\.!]*[.!])"
        ],
        page_text
    )


# ----------------------------
# MAIN SCRAPER
# ----------------------------
def scrape_td_card(driver, card_cfg):
    url = card_cfg["url"]
    card_id = card_cfg["card_id"]

    html, soup, page_text = fetch_page_text(driver, url)

    card_name = extract_generic_td_card_name(driver, soup, page_text)
    issuer = "TD"
    country = "Canada"
    network = "Visa" if "visa" in page_text.lower() else ""
    card_type = "Credit Card"

    annual_fee = extract_td_annual_fee(page_text)
    purchase_rate = extract_td_purchase_rate(page_text)
    cash_advance_rate = extract_td_cash_advance_rate(page_text)
    additional_card_fee = extract_td_additional_card_fee(page_text)
    additional_card_fee_detail = extract_td_additional_card_fee_detail(page_text)

    welcome_bonus_summary = extract_td_welcome_bonus(page_text)
    eligibility_summary = extract_td_eligibility(page_text)
    rewards_summary = extract_td_rewards_summary(page_text)
    redemption_summary = extract_td_redemption_summary(page_text)

    best_for = infer_best_for(page_text)
    one_liner = infer_one_liner(page_text)

    overview_text = extract_between(
        page_text,
        start_markers=["TD Cash Back Visa Infinite Card", "TD Cash Back Visa Infinite"],
        end_markers=["Eligibility Requirements:", "At a glance"],
        max_chars=1800
    )

    eligibility_text = extract_between(
        page_text,
        start_markers=["Eligibility Requirements:"],
        end_markers=["Special Offer", "At a glance"],
        max_chars=1200
    )

    at_a_glance_text = extract_between(
        page_text,
        start_markers=["At a glance"],
        end_markers=["It's easy to earn and redeem Cash Back Dollars", "Perks and offers"],
        max_chars=1400
    )

    rewards_text = extract_between(
        page_text,
        start_markers=["It's easy to earn and redeem Cash Back Dollars"],
        end_markers=["Perks and offers", "More Card Benefits and Features"],
        max_chars=2200
    )

    perks_text = extract_between(
        page_text,
        start_markers=["Perks and offers"],
        end_markers=["More Card Benefits and Features", "Frequently asked questions"],
        max_chars=1400
    )

    # Best chunks for vector DB
    benefits_parent_text = extract_between(
        page_text,
        start_markers=["More Card Benefits and Features"],
        end_markers=["Frequently asked questions", "Why should you get the TD Cash Back Visa Infinite Credit Card?"],
        max_chars=3000
    )

    visa_infinite_text = extract_between(
        page_text,
        start_markers=["Visa Infinite Benefits"],
        end_markers=["Automotive Benefits"],
        max_chars=1500
    )

    automotive_text = extract_between(
        page_text,
        start_markers=["Automotive Benefits"],
        end_markers=["Travel Benefits"],
        max_chars=1500
    )

    travel_text = extract_between(
        page_text,
        start_markers=["Travel Benefits"],
        end_markers=["Additional Benefits & Features"],
        max_chars=1500
    )

    additional_features_text = extract_between(
        page_text,
        start_markers=["Additional Benefits & Features"],
        end_markers=["Security"],
        max_chars=1500
    )

    security_text = extract_between(
        page_text,
        start_markers=["Security"],
        end_markers=["Optional benefits"],
        max_chars=1500
    )

    optional_benefits_text = extract_between(
        page_text,
        start_markers=["Optional benefits"],
        end_markers=["Frequently asked questions", "Why should you get the TD Cash Back Visa Infinite Credit Card?"],
        max_chars=1500
    )

    faq_why_text = extract_between(
        page_text,
        start_markers=["Why should you get the TD Cash Back Visa Infinite Credit Card?"],
        end_markers=["How do you apply for this Cash Back credit card?"],
        max_chars=1200
    )

    faq_apply_text = extract_between(
        page_text,
        start_markers=["How do you apply for this Cash Back credit card?"],
        end_markers=["How do you earn Cash Back on this TD Cash Back Credit Card?"],
        max_chars=1000
    )

    faq_earn_text = extract_between(
        page_text,
        start_markers=["How do you earn Cash Back on this TD Cash Back Credit Card?"],
        end_markers=["How do you redeem Cash Back dollars?"],
        max_chars=1000
    )

    faq_redeem_text = extract_between(
        page_text,
        start_markers=["How do you redeem Cash Back dollars?"],
        end_markers=["Find out more about our credit cards", "Legal"],
        max_chars=1000
    )

    fees_text = clean_text(
        f"Annual fee: ${annual_fee}; "
        f"Purchase interest: {purchase_rate}; "
        f"Cash advance interest: {cash_advance_rate}; "
        f"Additional cardholder fee: ${additional_card_fee}. "
        f"{additional_card_fee_detail}"
    )

    card_record = {
        "card_id": card_id,
        "card_name": card_name,
        "issuer": issuer,
        "country": country,
        "network": network,
        "card_type": card_type,
        "link": url,
        "monthly_fee": "",
        "annual_fee": annual_fee,
        "welcome_bonus_summary": welcome_bonus_summary,
        "best_for": best_for,
        "one_liner": one_liner,
        "eligibility_summary": eligibility_summary,
        "purchase_rate": purchase_rate,
        "cash_advance_rate": cash_advance_rate,
        "additional_card_fee": additional_card_fee,
        "additional_card_fee_detail": additional_card_fee_detail,
        "rewards_summary": rewards_summary,
        "redemption_summary": redemption_summary
    }

    snippets = [
        {"chunk_id": f"{card_id}_overview", "card_id": card_id, "section": "overview", "text": overview_text},
        {"chunk_id": f"{card_id}_eligibility", "card_id": card_id, "section": "eligibility", "text": eligibility_text},
        {"chunk_id": f"{card_id}_at_a_glance", "card_id": card_id, "section": "at_a_glance", "text": at_a_glance_text},
        {"chunk_id": f"{card_id}_welcome_bonus", "card_id": card_id, "section": "welcome_bonus", "text": welcome_bonus_summary},
        {"chunk_id": f"{card_id}_rewards", "card_id": card_id, "section": "rewards", "text": rewards_text},
        {"chunk_id": f"{card_id}_perks", "card_id": card_id, "section": "perks_and_offers", "text": perks_text},
        {"chunk_id": f"{card_id}_benefits_parent", "card_id": card_id, "section": "more_card_benefits_and_features", "text": benefits_parent_text},
        {"chunk_id": f"{card_id}_visa_infinite", "card_id": card_id, "section": "visa_infinite_benefits", "text": visa_infinite_text},
        {"chunk_id": f"{card_id}_automotive", "card_id": card_id, "section": "automotive_benefits", "text": automotive_text},
        {"chunk_id": f"{card_id}_travel", "card_id": card_id, "section": "travel_benefits", "text": travel_text},
        {"chunk_id": f"{card_id}_additional_features", "card_id": card_id, "section": "additional_features", "text": additional_features_text},
        {"chunk_id": f"{card_id}_security", "card_id": card_id, "section": "security", "text": security_text},
        {"chunk_id": f"{card_id}_optional", "card_id": card_id, "section": "optional_benefits", "text": optional_benefits_text},
        {"chunk_id": f"{card_id}_faq_why", "card_id": card_id, "section": "faq_why_get_this_card", "text": faq_why_text},
        {"chunk_id": f"{card_id}_faq_apply", "card_id": card_id, "section": "faq_apply", "text": faq_apply_text},
        {"chunk_id": f"{card_id}_faq_earn", "card_id": card_id, "section": "faq_earn", "text": faq_earn_text},
        {"chunk_id": f"{card_id}_faq_redeem", "card_id": card_id, "section": "faq_redeem", "text": faq_redeem_text},
        {"chunk_id": f"{card_id}_fees", "card_id": card_id, "section": "fees", "text": fees_text},
    ]

    snippets = [s for s in snippets if s["text"]]

    return card_record, snippets


# ----------------------------
# RUN
# ----------------------------
def main():
    driver = get_driver(headless=True)

    all_cards = []
    all_snippets = []

    try:
        for card in TD_CARDS:
            print(f"Scraping: {card['card_id']}")
            card_record, snippets = scrape_td_card(driver, card)

            print("Extracted card name:", card_record["card_name"])

            all_cards.append(card_record)
            all_snippets.extend(snippets)
            time.sleep(2)
    finally:
        driver.quit()

    cards_df = pd.DataFrame(all_cards)
    cards_df.to_csv(CSV_PATH, index=False)

    with open(SNIPPETS_PATH, "w", encoding="utf-8") as f:
        for item in all_snippets:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

    print("Saved CSV to:", CSV_PATH)
    print("Saved snippets to:", SNIPPETS_PATH)
    print(cards_df.T)

    print("\nSnippet preview:")
    for s in all_snippets:
        print(f"- {s['section']}: {s['text'][:220]}...")


if __name__ == "__main__":
    main()

Scraping: td_cash_back_visa_infinite_ca
Extracted card name: TD Cash Back Visa Infinite Card
Scraping: td_cash_back_visa_ca
Extracted card name: Apply for a TD Cash Back Visa Card
Scraping: td_aeroplan_visa_platinum_card_ca
Extracted card name: TD Aeroplan Visa Platinum Credit Card
Scraping: td_aeroplan_visa_infinite_privilege_card_ca
Extracted card name: TD Aeroplan Visa Platinum Credit Card
Scraping: td_first_class_travel_visa_infinite_card_ca
Extracted card name: TD First Class Travel Visa Infinite Card
Scraping: td_platinum_travel_visa_card_ca
Extracted card name: TD Platinum Travel Visa Card
Scraping: td_rewards_visa_card_ca
Extracted card name: TD Rewards Visa Credit Card
Scraping: td_low_rate_visa_card_ca
Extracted card name: TD Low Rate Visa Credit Card
Scraping: td_us_dollar_visa_card_ca
Extracted card name: Apply for a TD U.S. Dollar Visa Card
Scraping: td_business_travel_visa_card_ca
Extracted card name: Apply for a TD Business Travel Visa Card - TD Canada Trust
Scraping: td